In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

n_records = 10000

# 1. Base Framework Setup
tx_ids = [f"TXN-{1000000 + i}" for i in range(n_records)]
start_date = datetime(2026, 9, 22)
timestamps = [start_date + timedelta(
    days=random.randint(0, 29), hours=random.randint(0, 23),
    minutes=random.randint(0, 59), seconds=random.randint(0, 59)
) for _ in range(n_records)]
timestamps.sort()

agents = {
    "AGNT-FIN-01": {"name": "PortfolioRebalancer", "type": "DeFi / Investment", "avg_amt": 1250, "std_amt": 400},
    "AGNT-UTIL-02": {"name": "AutoUtilityOptimizer", "type": "SaaS / Bills", "avg_amt": 45, "std_amt": 15},
    "AGNT-SHOP-03": {"name": "ProcurementBot", "type": "Inventory Supply", "avg_amt": 450, "std_amt": 150},
    "AGNT-AD-04": {"name": "AdSpendOptimizer", "type": "Marketing", "avg_amt": 2100, "std_amt": 700},
    "AGNT-API-05": {"name": "LLM-MeteredPay", "type": "API / Infrastructure", "avg_amt": 8, "std_amt": 4},
}

agent_keys = list(agents.keys())
chosen_agent_keys = np.random.choice(agent_keys, size=n_records, p=[0.2, 0.25, 0.15, 0.1, 0.3])

amounts, agent_names, agent_types = [], [], []
for k in chosen_agent_keys:
    agent_names.append(agents[k]["name"])
    agent_types.append(agents[k]["type"])
    amt = max(0.01, round(np.random.normal(agents[k]["avg_amt"], agents[k]["std_amt"]), 2))
    amounts.append(amt)

# 2. Add Base Infrastructure Columns
ip_pool = ["192.168.1.50", "12.44.192.103", "45.79.10.4", "185.220.101.5", "104.244.42.1"]
originating_ips = np.random.choice(ip_pool, size=n_records, p=[0.5, 0.2, 0.15, 0.1, 0.05])
api_versions = np.random.choice(["v2.0-stable", "v2.1-beta", "v1.8-legacy"], size=n_records, p=[0.7, 0.2, 0.1])
server_regions = np.random.choice(["us-east-1", "eu-central-1", "ap-northeast-1"], size=n_records, p=[0.6, 0.3, 0.1])

triggers = {
    "DeFi / Investment": ["Threshold Trigger: Asset Dip", "Scheduled Portfolio Rebalance", "Direct Instruction Override"],
    "SaaS / Bills": ["Dynamic Rate Drop Found", "Usage Threshold Reached", "Contract Renewal Auto-Negotiated"],
    "Inventory Supply": ["Stock Level < 15%", "Just-In-Time Restock Signal", "Bulk Discount Match"],
    "Marketing": ["ROAS Target Exceeded", "Bid Cap Adjustment", "Budget Pacing Redistribution"],
    "API / Infrastructure": ["Token Pool Replenishment", "Compute Node Scaling", "Rapid Request Reset"]
}
trigger_reasons = [random.choice(triggers[t]) for t in agent_types]

# Construct Base DataFrame
df = pd.DataFrame({
    "Transaction_ID": tx_ids,
    "Timestamp": timestamps,
    "Agent_ID": chosen_agent_keys,
    "Agent_Name": agent_names,
    "Agent_Type": agent_types,
    "Amount_USD": amounts,
    "Trigger_Reason": trigger_reasons,
    "Originating_IP": originating_ips,
    "API_Version": api_versions,
    "Server_Region": server_regions,
    "Payment_Rail": np.random.choice(["Stripe Link", "FedNow", "USDC Stablecoin"], size=n_records),
    "Is_Anomalous_Fraud": 0
})

# 3. Controlled Noise & Overlapping Fraud Injection (Targeting exactly ~3.2% total fraud)
# Scenario A: Velocity Spike (Injecting 200 rows)
velocity_fraud_indices = df[df["Agent_ID"] == "AGNT-API-05"].sample(n=200, random_state=42).index
df.loc[velocity_fraud_indices, "Is_Anomalous_Fraud"] = 1
df.loc[velocity_fraud_indices, "Originating_IP"] = "185.220.101.5"  # Blends with existing legitimate traffic from this IP
df.loc[velocity_fraud_indices, "Trigger_Reason"] = "Token Pool Replenishment"  # Masked as a real trigger

# Scenario B: High-Value Prompt Injection (Injecting 120 rows)
high_val_indices = df[(df["Agent_ID"] == "AGNT-FIN-01")].sample(n=120, random_state=42).index
df.loc[high_val_indices, "Is_Anomalous_Fraud"] = 1
df.loc[high_val_indices, "Amount_USD"] = np.random.uniform(25000, 40000, size=120).round(2)
df.loc[high_val_indices, "Trigger_Reason"] = "Direct Instruction Override"

# 4. Inject Noise into Legitimate Data to Break Correlation
# Add high-value transactions to legitimate data so high numbers don't instantly equal fraud
legit_high_val_indices = df[(df["Is_Anomalous_Fraud"] == 0) & (df["Agent_ID"] == "AGNT-FIN-01")].sample(n=150, random_state=42).index
df.loc[legit_high_val_indices, "Amount_USD"] = np.random.uniform(20000, 38000, size=150).round(2)

# Make legitimate transactions occasionally use the suspicious IP or trigger code
legit_noise_indices = df[df["Is_Anomalous_Fraud"] == 0].sample(n=400, random_state=42).index
df.loc[legit_noise_indices, "Originating_IP"] = "185.220.101.5"

# Convert timestamps for presentation
df["Timestamp"] = df["Timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Export to CSV
df.to_csv("noisy_agentic_payments_10k.csv", index=False)
fraud_percent = (df["Is_Anomalous_Fraud"].sum() / len(df)) * 100
print(f"Dataset generated! Total fraud instances: {df['Is_Anomalous_Fraud'].sum()} ({fraud_percent:.1f}% of total).")


Dataset generated! Total fraud instances: 320 (3.2% of total).
